In [1]:
!pip install pandas
!pip install ipyaggrid

import pandas as pd
from ipyaggrid import Grid



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Load the Stata dataset
data = pd.read_stata("E:\\UofT\\03_ML\\Project\\project_file\\data\\AEJApp-20090168_data.dta")
print(data.head())
data["salary_04"] = data["salary_04"] / 1000
data["salary_06"] = data["salary_06"] / 1000

   age_s  dmarried_s  empl_06  salary_06  profit_06  tenure_06  days_06  \
0   22.0         0.0      1.0        0.0   240000.0  15.233334     22.0   
1   22.0         0.0      1.0   116000.0        0.0   1.866667     10.0   
2   24.0         0.0      1.0   650000.0        0.0   1.866667     28.0   
3   24.0         0.0      1.0   408000.0        0.0   0.100000     28.0   
4   22.0         0.0      0.0        0.0        0.0   0.000000      0.0   

   hours_06  contract_06  dformal_06  ...  ldays_04  lhours_04   select  \
0      84.0          0.0         0.0  ...  3.178054   4.094345  control   
1      14.0          0.0         0.0  ...  2.708050   3.401197  control   
2      91.0          0.0         0.0  ...  3.401197   4.248495  control   
3      48.0          0.0         0.0  ...       NaN        NaN  control   
4       0.0          0.0         0.0  ...       NaN        NaN  control   

   pempl_06  pempl_04  dcontinue  codigo_ecap  codigo_curs  dwomen  p_selecap  
0       0.0       

In [19]:
# Define pre/post variable groups
var_map = {
    "employment": ["empl_04", "empl_06"],
    "salary": ["salary_04", "salary_06"],
    "paid employment": ["pempl_04", "pempl_06"],
    "age": ["age_lb", "age_s"],
    "education": ["educ_lb", "educ_s"],
    "married": ["dmarried_lb", "dmarried_s"],
    "gender": ["dwomen", "dwomen"]

}

def make_summary(df):
    frames = []
    for label, (pre, post) in var_map.items():
        # Compute summary statistics
        temp = df[[pre, post]].agg(["mean", "median", "std", "min", "max"])
        temp = temp.T.reset_index().rename(columns={"index": "Variable"})
        
        # Replace column names with label
        temp["Variable"] = label
        
        # Assign Pre/Post periods
        temp["Period"] = ["Pre", "Post"]
        
        frames.append(temp)

    # Combine all blocks
    result = pd.concat(frames, ignore_index=True)

    # Reorder columns
    result = result[["Variable", "Period", "mean", "median", "std", "min", "max"]]

    return result.round(2)


# -------------------------------
# Split by treatment status
# select == 1   → treated
# select == 0   → control
# -------------------------------

control_summary = make_summary(data[data["select"] == "control"])
treated_summary = make_summary(data[data["select"] == "selected"])

# Print clean results
print("\n=== Control Group (select = 0) ===")
display(control_summary)

print("\n=== Treated Group (select = 1) ===")
display(treated_summary)


=== Control Group (select = 0) ===


,Variable,Period,mean,median,std,min,max
0,employment,Pre,0.500000,1.0,0.500000,0.0,1.0
1,employment,Post,0.720000,1.0,0.450000,0.0,1.0
2,salary,Pre,99.660004,0.0,156.179993,0.0,1000.0
3,salary,Post,215.789993,204.0,203.110001,0.0,1000.0
4,paid employment,Pre,0.350000,0.0,0.480000,0.0,1.0
5,paid employment,Post,0.610000,1.0,0.490000,0.0,1.0
6,age,Pre,21.230000,21.0,2.030000,18.0,26.0
7,age,Post,22.780001,23.0,2.050000,19.0,28.0
8,education,Pre,9.990000,11.0,1.910000,0.0,16.0
9,education,Post,10.210000,11.0,1.770000,2.0,15.0



=== Treated Group (select = 1) ===


,Variable,Period,mean,median,std,min,max
0,employment,Pre,0.540000,1.0,0.500000,0.0,1.0
1,employment,Post,0.760000,1.0,0.430000,0.0,1.0
2,salary,Pre,105.669998,0.0,155.539993,0.0,800.0
3,salary,Post,255.490005,300.0,221.490005,0.0,2000.0
4,paid employment,Pre,0.390000,0.0,0.490000,0.0,1.0
5,paid employment,Post,0.670000,1.0,0.470000,0.0,1.0
6,age,Pre,21.080000,21.0,2.050000,18.0,29.0
7,age,Post,22.670000,22.0,2.120000,19.0,46.0
8,education,Pre,10.170000,11.0,1.670000,2.0,14.0
9,education,Post,10.400000,11.0,1.540000,2.0,15.0


In [20]:
control_summary.to_csv("control_summary.csv", index=False)
treated_summary.to_csv("treated_summary.csv", index=False)

In [21]:
import pandas as pd

# Load your summaries
control = pd.read_csv("control_summary.csv")
treat = pd.read_csv("treated_summary.csv")

# Combine and reshape for LaTeX
control["Group"] = "Control"
treat["Group"] = "Treatment"

df = pd.concat([control, treat])

# Sort for clean ordering
df = df.sort_values(["Variable", "Period", "Group"])

# Generate LaTeX rows
latex_rows = []
for var in df["Variable"].unique():
    latex_rows.append(f"\\textbf{{{var.replace('_',' ').title()}}} \\\\ ")
    sub = df[df["Variable"] == var]
    for _, r in sub.iterrows():
        row = (
            f" & {r['Period']} "
            f" & {r['Group']} "
            f" & ${r['mean']}$ "
            f" & ${r['median']}$ "
            f" & ${r['std']}$ \\\\ "
        )
        latex_rows.append(row)
    latex_rows.append("\\hline")

# Save to file
with open("desc_table.tex", "w") as f:
    for line in latex_rows:
        f.write(line + "\n")

print("LaTeX table saved to desc_table.tex")


LaTeX table saved to desc_table.tex


In [24]:
import numpy as np
import pandas as pd

# Gender indicator
data["gender"] = np.where(data["dwomen"] == 1, "Women", "Men")

# Variables to summarize
var_map = {
    "employment": ["empl_04", "empl_06"],
    "salary": ["salary_04", "salary_06"],
    "paid employment": ["pempl_04", "pempl_06"],
    "age": ["age_lb", "age_s"],
    "education": ["educ_lb", "educ_s"],
    "married": ["dmarried_lb", "dmarried_s"]
}

def make_gender_summary(df):
    frames = []
    for label, (pre, post) in var_map.items():
        
        # PRE mean by gender
        pre_means = df.groupby("gender")[pre].mean()
        
        # POST mean by gender
        post_means = df.groupby("gender")[post].mean()
        
        # Construct rows
        frames.append([label, "Pre", "Women", round(pre_means["Women"], 3)])
        frames.append([label, "Pre", "Men",   round(pre_means["Men"], 3)])
        frames.append([label, "Post", "Women", round(post_means["Women"], 3)])
        frames.append([label, "Post", "Men",   round(post_means["Men"], 3)])

    result = pd.DataFrame(frames, columns=["Variable", "Period", "Gender", "Mean"])
    return result

final_summary = make_gender_summary(data)

display(final_summary)

# Save to CSV if needed
final_summary.to_csv("summary_gender_only.csv", index=False)


,Variable,Period,Gender,Mean
0,employment,Pre,Women,0.464000
1,employment,Pre,Men,0.581000
2,employment,Post,Women,0.671000
3,employment,Post,Men,0.830000
4,salary,Pre,Women,86.180000
5,salary,Pre,Men,122.170998
6,salary,Post,Women,196.300003
7,salary,Post,Men,285.446991
8,paid employment,Pre,Women,0.339000
9,paid employment,Pre,Men,0.400000
